# Milestone 1: Data Profiling (Time Series)

Profil awal dataset **PEMS-BAY** (±16,9 juta baris, 3 file Parquet) menggunakan Polars LazyFrame dan DuckDB. Dataset ini berisi kecepatan lalu lintas dari 325 sensor jalan raya di San Francisco Bay Area dengan resolusi 5 menit, sehingga cocok untuk analisis time series dan spasial pada milestone berikutnya. Sumber serta deskripsi dataset ada pada `data/README.md`.

## Unduh Dataset

Dataset diunduh ke disk lokal dengan `snapshot_download`, bukan dibaca langsung lewat `hf://`:

1. **Simpan ke `data/raw/pems-bay/`** supaya tidak perlu mengunduh ulang setiap kernel restart.
2. **Hanya file yang dibutuhkan**: tiga split Parquet (`train`, `val`, `test`) `sensor_graph/sensor_locations.csv` (koordinat sensor), dan `sensor_graph/adj_mx_mapping.json` (pemetaan `node_id` ke `sensor_id`).
3. **Resume/skip**: jika semua file sudah ada dan ukurannya > 0 byte, unduhan dilewati sehingga cell ini idempotent.
4. **Verifikasi akhir**: jumlah file dan total ukuran dicek ulang sebelum profiling.

In [1]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pems-bay'
DATA_GLOB = (RAW_DIR / '*.parquet').as_posix()
LOKASI_CSV = RAW_DIR / 'sensor_graph' / 'sensor_locations.csv'

REPO_ID = 'witgaw/PEMS-BAY'
BERKAS = ['train.parquet', 'val.parquet', 'test.parquet', 'sensor_graph/sensor_locations.csv', 'sensor_graph/adj_mx_mapping.json']


def file_valid():
    return [b for b in BERKAS if (RAW_DIR / b).exists() and (RAW_DIR / b).stat().st_size > 0]


if len(file_valid()) < len(BERKAS):
    from huggingface_hub import snapshot_download

    snapshot_download(REPO_ID, repo_type='dataset', local_dir=str(RAW_DIR),
                      allow_patterns=BERKAS, max_workers=1)

file_parquet = sorted(RAW_DIR.glob('*.parquet'))
total_mb = sum(f.stat().st_size for f in file_parquet) / 1024**2
print(f'File Parquet : {len(file_parquet)} / 3')
print(f'Total ukuran : {total_mb:.1f} MB')
print(f'Berkas sensor: {len(file_valid()) - 3} / 2 (lokasi + pemetaan)')

C:\Users\Huawei\AppData\Local\Temp\claude\d--SEMESTER-7-BIG-DATA\53f4d214-756e-413a-89ad-373a59c5ffc3\scratchpad\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:  20%|██        | 1/5 [00:00<00:03,  1.16it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00,  5.61it/s]

File Parquet : 3 / 3
Total ukuran : 491.2 MB
Berkas sensor: 2 / 2 (lokasi + pemetaan)


In [2]:
import duckdb
import polars as pl

if len(file_parquet) < 3:
    raise FileNotFoundError(f'Dataset tidak ditemukan di {RAW_DIR}. Jalankan cell unduh di atas.')

print(f'Pola data: {DATA_GLOB}')

Pola data: D:/SEMESTER 7/BIG DATA/tugas1-abd-2026/data/raw/pems-bay/*.parquet


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil. Skema berisi 50 kolom: `node_id`, `t0_timestamp`, serta 24 kolom input `x_t-11 ... x_t+0` dan 24 kolom target `y_t+1 ... y_t+12`, masing-masing dengan dimensi `d0` (kecepatan) dan `d1` (waktu dalam hari, 0-1).

In [3]:
lazy_data = pl.scan_parquet(DATA_GLOB)
skema = lazy_data.collect_schema()

print(f'Jumlah kolom: {len(skema)}')
for nama, tipe in list(skema.items())[:4]:
    print(f'  {nama:<20} {tipe}')
print('  ...')
for nama, tipe in list(skema.items())[-2:]:
    print(f'  {nama:<20} {tipe}')

tipe_hitung = pl.Series(list(map(str, skema.values()))).value_counts().sort('count', descending=True)
print()
print(tipe_hitung)

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.col('node_id').n_unique().alias('sensor_unik'),
    pl.col('t0_timestamp').n_unique().alias('timestamp_unik'),
    pl.sum_horizontal(pl.all().null_count()).alias('total_null'),
]).collect(engine='streaming')
ringkasan

Jumlah kolom: 50
  node_id              Int64
  t0_timestamp         String
  x_t-11_d0            Float64
  x_t-11_d1            Float64
  ...
  y_t+12_d0            Float64
  y_t+12_d1            Float64

shape: (3, 2)
┌─────────┬───────┐
│         ┆ count │
│ ---     ┆ ---   │
│ str     ┆ u32   │
╞═════════╪═══════╡
│ Float64 ┆ 48    │
│ String  ┆ 1     │
│ Int64   ┆ 1     │
└─────────┴───────┘


jumlah_baris,sensor_unik,timestamp_unik,total_null
u32,u32,u32,u32
16930225,325,52093,0


In [4]:
contoh_data = lazy_data.select(
    'node_id', 't0_timestamp', 'x_t-11_d0', 'x_t+0_d0', 'x_t+0_d1', 'y_t+1_d0', 'y_t+12_d0'
).head(10).collect(engine='streaming')
contoh_data

node_id,t0_timestamp,x_t-11_d0,x_t+0_d0,x_t+0_d1,y_t+1_d0,y_t+12_d0
i64,str,f64,f64,f64,f64,f64
0,"""2017-05-25T18:45:00""",70.6,73.1,0.78125,70.9,71.8
1,"""2017-05-25T18:45:00""",19.5,22.5,0.78125,24.1,58.6
2,"""2017-05-25T18:45:00""",56.0,56.7,0.78125,54.4,59.9
3,"""2017-05-25T18:45:00""",25.5,32.3,0.78125,32.6,55.2
4,"""2017-05-25T18:45:00""",58.7,58.0,0.78125,62.5,62.2
5,"""2017-05-25T18:45:00""",66.5,68.1,0.78125,68.7,67.5
6,"""2017-05-25T18:45:00""",11.5,52.6,0.78125,57.3,63.2
7,"""2017-05-25T18:45:00""",27.5,51.9,0.78125,51.9,61.6
8,"""2017-05-25T18:45:00""",13.9,35.3,0.78125,40.7,58.3


## Profil Dimensi Waktu

`t0_timestamp` bertipe String (ISO 8601), jadi di-parse ke `Datetime` terlebih dahulu. Pemeriksaan: rentang waktu tiap split (harus berurutan kronologis tanpa tumpang tindih), jumlah sensor per timestamp, dan interval antar timestamp (seharusnya 5 menit).

In [5]:
waktu = lazy_data.select(
    pl.col('t0_timestamp').str.to_datetime('%Y-%m-%dT%H:%M:%S').alias('t0'),
    pl.col('node_id'),
)

rentang_split = pl.concat([
    pl.scan_parquet(RAW_DIR / f'{s}.parquet')
    .select(
        pl.lit(s).alias('split'),
        pl.len().alias('jumlah_baris'),
        pl.col('t0_timestamp').str.to_datetime('%Y-%m-%dT%H:%M:%S').min().alias('mulai'),
        pl.col('t0_timestamp').str.to_datetime('%Y-%m-%dT%H:%M:%S').max().alias('akhir'),
    )
    .collect(engine='streaming')
    for s in ['train', 'val', 'test']
])
rentang_split

split,jumlah_baris,mulai,akhir
str,u32,datetime[μs],datetime[μs]
"""train""",11851125,2017-01-01 00:55:00,2017-05-07 16:35:00
"""val""",1692925,2017-05-07 16:40:00,2017-05-25 18:40:00
"""test""",3386175,2017-05-25 18:45:00,2017-06-30 22:55:00


In [6]:
per_timestamp = (
    waktu.group_by('t0').agg(pl.col('node_id').n_unique().alias('jumlah_sensor'))
    .sort('t0')
    .collect(engine='streaming')
)

selisih = per_timestamp.select(pl.col('t0').diff().alias('selisih')).drop_nulls()
print(f'Timestamp unik          : {per_timestamp.height:,}')
print(f'Sensor per timestamp    : min {per_timestamp["jumlah_sensor"].min()}, maks {per_timestamp["jumlah_sensor"].max()}')
print('Distribusi interval antar timestamp:')
print(selisih['selisih'].value_counts().sort('count', descending=True).head(5))

Timestamp unik          : 52,093
Sensor per timestamp    : min 325, maks 325
Distribusi interval antar timestamp:
shape: (2, 2)
┌──────────────┬───────┐
│ selisih      ┆ count │
│ ---          ┆ ---   │
│ duration[μs] ┆ u32   │
╞══════════════╪═══════╡
│ 5m           ┆ 52091 │
│ 1h 5m        ┆ 1     │
└──────────────┴───────┘


## Profil Nilai Kecepatan

Dimensi `d0` adalah kecepatan lalu lintas (mph). Pada PEMS-BAY, nilai `0` umumnya menandai pembacaan sensor yang hilang, bukan kecepatan nol sebenarnya, jadi proporsinya dicek terpisah dari `null`.

In [7]:
kolom_d0 = [c for c in skema if c.endswith('_d0')]

agregat = lazy_data.select(
    *[(pl.col(c) == 0).sum().alias(f'nol|{c}') for c in kolom_d0],
    *[pl.col(c).filter(pl.col(c) > 0).sum().alias(f'jml|{c}') for c in kolom_d0],
    *[(pl.col(c) > 0).sum().alias(f'pos|{c}') for c in kolom_d0],
    *[pl.col(c).filter(pl.col(c) > 0).min().alias(f'min|{c}') for c in kolom_d0],
    *[pl.col(c).max().alias(f'maks|{c}') for c in kolom_d0],
).collect(engine='streaming').row(0, named=True)


def total(awalan, fungsi=sum):
    return fungsi(v for k, v in agregat.items() if k.startswith(awalan + '|'))


n_nilai = ringkasan['jumlah_baris'].item() * len(kolom_d0)
statistik_d0 = pl.DataFrame({
    'jumlah_nilai': [n_nilai],
    'nol': [total('nol')],
    'persen_nol': [round(total('nol') / n_nilai * 100, 3)],
    'min_positif': [total('min', min)],
    'maks': [total('maks', max)],
    'rata_rata_positif': [round(total('jml') / total('pos'), 2)],
})
statistik_d0

jumlah_nilai,nol,persen_nol,min_positif,maks,rata_rata_positif
i64,i64,f64,f64,f64,f64
406325400,12504,0.003,3.0,85.1,62.62


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik kolom kunci (kolom lain berstruktur sama sehingga tidak perlu ditampilkan semuanya).

In [8]:
connection = duckdb.connect()
profil_duckdb = connection.sql(f'''
    SUMMARIZE SELECT node_id, t0_timestamp, "x_t-11_d0", "x_t+0_d0", "x_t+0_d1", "y_t+1_d0", "y_t+12_d0"
    FROM read_parquet('{DATA_GLOB}')
''').pl()
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""node_id""","""BIGINT""","""0""","""324""",325,"""162.0""","""93.81897739742477""","""81""","""162""","""243""",16930225,0.00
"""t0_timestamp""","""VARCHAR""","""2017-01-01T00:55:00""","""2017-06-30T22:55:00""",52707,null,null,null,null,null,16930225,0.00
"""x_t-11_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.617724991840085""","""9.595799942893562""","""62.127141242141484""","""65.27655443051354""","""67.48952485740938""",16930225,0.00
"""x_t+0_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.61742577549779""","""9.595823894090481""","""62.129349158263246""","""65.28371810991075""","""67.48026169082385""",16930225,0.00
"""x_t+0_d1""","""DOUBLE""","""0.0""","""0.9965277777777778""",286,"""0.49834624027113866""","""0.2885314643942237""","""0.24873277496575902""","""0.49802374317862397""","""0.7486746955291619""",16930225,0.00
"""y_t+1_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.61740679760472""","""9.595818415003768""","""62.129348636455795""","""65.27247300340268""","""67.48988292140899""",16930225,0.00
"""y_t+12_d0""","""DOUBLE""","""0.0""","""85.1""",798,"""62.617184254785435""","""9.595760446356193""","""62.13050528212742""","""65.2686499624394""","""67.48483442672084""",16930225,0.00


## Profil Lokasi Sensor

`sensor_locations.csv` memuat koordinat tiap sensor. Berkas ini menjadi dasar analisis spasial pada milestone berikutnya.

In [9]:
lokasi = pl.read_csv(LOKASI_CSV)
print(f'Jumlah sensor di berkas lokasi: {lokasi.height}')
print(f'Sensor unik                   : {lokasi["sensor_id"].n_unique()}')
print(f'Null                          : {lokasi.null_count().to_numpy().sum()}')
lokasi.describe()

Jumlah sensor di berkas lokasi: 325
Sensor unik                   : 325
Null                          : 0


statistic,sensor_id,latitude,longitude
str,f64,f64,f64
"""count""",325.0,325.0,325.0
"""null_count""",0.0,0.0,0.0
"""mean""",402802.818462,37.344741,-121.940083
"""std""",3065.18692,0.050665,0.070151
"""min""",400001.0,37.250424,-122.079539
"""25%""",400582.0,37.316628,-121.996301
"""50%""",401597.0,37.344411,-121.921627
"""75%""",404453.0,37.388372,-121.88076
"""max""",414694.0,37.427486,-121.840267


## Catatan Milestone 1

1. Dataset berisi 16.930.225 baris dan 50 kolom dalam 3 file Parquet (`train`, `val`, `test`) dengan total ±491 MiB (±515 MB). Ukuran file sedikit di bawah 500 MiB, sehingga syarat ukuran terpenuhi lewat kriteria lebih dari 1 juta baris.
2. Kolom terdiri dari `node_id` (Int64), `t0_timestamp` (String ISO 8601), dan 48 kolom Float64: 24 kolom input `x_t-11 ... x_t+0` dan 24 kolom target `y_t+1 ... y_t+12`. Tiap kolom punya dimensi `d0` (kecepatan, mph) dan `d1` (waktu dalam hari, 0-1). Satu baris mewakili satu sensor pada satu waktu referensi `t0`, dengan jendela 12 langkah ke belakang dan 12 langkah ke depan. Baris yang berurutan berbagi nilai pada jendelanya.
3. Seluruh kolom bebas `null`. Kecepatan `0` menandai pembacaan sensor yang hilang dan muncul 12.504 kali dari 406.325.400 nilai `d0` (±0,003%). Milestone 2 menanganinya.
4. `t0_timestamp` mencakup 2017-01-01 00:55 sampai 2017-06-30 22:55 dengan 52.093 timestamp unik berresolusi 5 menit. Dari 52.092 selisih antar timestamp, 52.091 berjarak 5 menit dan satu berjarak 1 jam 5 menit. Selisih itu terjadi pada 2017-03-12 03:00 (lihat Milestone 2), bertepatan dengan pergantian waktu musim panas AS, jadi kemungkinan bukan data hilang.
5. Tiga split berurutan kronologis tanpa tumpang tindih: `train` (2017-01-01 sampai 2017-05-07), `val` (2017-05-07 sampai 2017-05-25), dan `test` (2017-05-25 sampai 2017-06-30). Pembagian ini aman dari kebocoran waktu.
6. Dataset mencakup 325 sensor (`node_id` 0-324), dan setiap timestamp memuat 325 sensor. `sensor_locations.csv` menyimpan 325 koordinat tanpa `null` (lintang 37,25 sampai 37,43; bujur -122,08 sampai -121,84). `adj_mx_mapping.json` memetakan `node_id` ke `sensor_id`.
7. Kecepatan positif berkisar 3,0 sampai 85,1 mph dengan rata-rata ±62,6 mph.
8. Untuk Milestone 2: ubah `t0_timestamp` ke `Datetime`, tangani kecepatan `0`, dan pakai `scan_parquet` dengan streaming supaya 16,9 juta baris tidak masuk ke memori sekaligus.